<a href="https://colab.research.google.com/github/aims-ai-research-foundations/pilot-workshop/blob/main/assignments/day3/day3-course4-student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A Transformer in 60 Minutes
## *The story of one sentence: "Jide cooked iyan with peppers"*

**Course 4 · AI Long Activity · Take-home notebook**

In this notebook you will trace one short Yoruba-English sentence through the inside of one attention layer. You will not train anything. You will not run a giant model. Instead, you will build the attention mechanism by hand on a five-token sentence so you can SEE what attention is doing.

By the end of this notebook you will:

1. Build word embeddings for five tokens and inspect their geometry.
2. Project them into Queries, Keys, and Values.
3. Compute attention scores by hand, apply softmax, produce the output.
4. Visualise attention as a heatmap and read what the model is paying attention to.
5. Apply a causal mask and watch the heatmap change.
6. Run a SECOND attention head and see how different heads learn different patterns.

If you can explain attention in plain words after this notebook, the lab worked.

## A note on the example sentence

If you came from the in-session walkthrough, you computed attention by hand on "Jide ate iyan" (3 tokens). This notebook extends that to "Jide cooked iyan with peppers" (5 tokens) for two reasons: a longer sentence makes the attention heatmap more interesting to read, and we need at least 4 distinct token roles for multi-head attention to demonstrate distinct patterns. The mechanism is identical; only the scale changes.

## How to use this notebook

Cells alternate between **read & run** (already filled in) and **`# TODO`** cells (you fill them in).

Protocol for every cell:

1. **Predict.** Before you run, ask yourself what number or shape you expect.
2. **Run.** Execute the cell.
3. **Explain.** If your prediction was wrong, write down (in the cell below) why.

If your prediction is right every time you are not stretching. Aim to be wrong at least twice.

**Total time:** about 60 minutes. **Setup:** Colab, no GPU needed.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

---
## The sentence we will trace

> *"Jide cooked iyan with peppers."*

Five tokens. Two of them (`iyan` and `peppers`) are food. One (`Jide`) is a person. One (`cooked`) is a verb. One (`with`) is a function word. Iyan, if you have not had it, is pounded yam.

We will give each token a 4-dimensional embedding. The dimensions are not magic; in a real model they are learned. For this notebook the dimensions were hand-picked so they roughly stand for:

- **dim 0**: food-ness
- **dim 1**: person-ness
- **dim 2**: action-ness
- **dim 3**: function-word-ness

This is a teaching simplification. In a real model you would never see dimensions this clean.

In [ ]:
tokens = ['Jide', 'cooked', 'iyan', 'with', 'peppers']

# 4D embeddings: rows are tokens, columns are [food, person, action, function]
X = np.array([
    [0.1, 0.9, 0.1, 0.1],   # Jide      -> mostly person
    [0.2, 0.1, 0.9, 0.2],   # cooked    -> mostly action
    [0.9, 0.1, 0.1, 0.2],   # iyan      -> mostly food
    [0.1, 0.1, 0.2, 0.9],   # with      -> mostly function
    [0.9, 0.1, 0.1, 0.3],   # peppers   -> mostly food
])
print('shape:', X.shape)
X

---
## Step 1. Embedding geometry

Before we touch attention, look at the embeddings on their own. The claim is that tokens with similar meaning end up close together in this 4D space. Let us check.

First, a similarity heatmap. Each cell shows the cosine similarity between two tokens. Bright = similar.

In [ ]:
def cosine_sim_matrix(M):
    """Pairwise cosine similarity for the rows of M."""
    norms = np.linalg.norm(M, axis=1, keepdims=True)
    M_unit = M / norms
    return M_unit @ M_unit.T

S = cosine_sim_matrix(X)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(S, cmap='viridis', vmin=0, vmax=1)
ax.set_xticks(range(len(tokens)), tokens, rotation=45, ha='right')
ax.set_yticks(range(len(tokens)), tokens)
ax.set_title('Cosine similarity between embeddings')
for i in range(len(tokens)):
    for j in range(len(tokens)):
        ax.text(j, i, f'{S[i,j]:.2f}', ha='center', va='center', color='white' if S[i,j] < 0.7 else 'black', fontsize=9)
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

**Observe.** Which two tokens are most similar? Does that match your intuition about the words?

### TODO 1. Compute the cosine similarity between two specific tokens, by hand-ish

Use NumPy to compute the cosine similarity between `iyan` and `peppers` directly (not from the matrix above).

Definition: cos(a, b) = (a · b) / (||a|| · ||b||).

In [ ]:
# TODO 1: compute cosine similarity between iyan (token 2) and peppers (token 4)
a = X[2]  # iyan
b = X[4]  # peppers

# Your code here. Replace the None.
cos_sim = None

print(f'cosine similarity between iyan and peppers: {cos_sim}')
# Expected: a number very close to 1.0

---
## Step 2. From embeddings to Queries, Keys, and Values

Self-attention does not compare embeddings to each other directly. It first projects each embedding into THREE different roles:

- **Query (Q):** what this token is *looking for*.
- **Key (K):** how this token *advertises itself*.
- **Value (V):** the actual *content* this token contributes if it gets selected.

Each role is produced by its own learned matrix: `W_Q`, `W_K`, `W_V`. In this notebook we will hand-craft them so the patterns are interpretable. Let `d_model = 4` (input dim) and `d_head = 2` (per-head dim).

The matrices below were designed so that:

- `Q` of a token will be HIGH if the token is an action (a verb): the verb is looking for arguments.
- `K` of a token will be HIGH on dim 0 if it is food, HIGH on dim 1 if it is a person.

So a verb's query should match nouns (food and person tokens) strongly.

In [ ]:
# Hand-crafted projections. d_model = 4, d_head = 2.

# W_Q: extracts 'action-ness' (dim 2) into BOTH Q dimensions
W_Q = np.array([
    [0.0, 0.0],
    [0.0, 0.0],
    [1.0, 1.0],
    [0.0, 0.0],
])

# W_K: dim 0 (food) -> K[0]; dim 1 (person) -> K[1]
W_K = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [0.0, 0.0],
    [0.0, 0.0],
])

# W_V: simple, keeps food-ness and person-ness as the value content
W_V = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [0.0, 0.0],
    [0.0, 0.0],
])

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

print('Q (queries) per token:'); print(Q)
print('\nK (keys) per token:'); print(K)
print('\nV (values) per token:'); print(V)

**Sanity check.** Look at `Q[1]` (the row for `cooked`). It should be the largest because cooked is a verb. Look at `K[0]` (Jide). The second component should be high (Jide is a person). Look at `K[2]` (iyan). The first component should be high (iyan is food).

---
## Step 3. Attention scores: who is looking at whom?

For every pair of tokens (i, j), we compute one number: how much should token i pay attention to token j? That number is the dot product of token i's query with token j's key.

All of these dot products at once is just a matrix multiply: `Q @ K.T`.

We also divide by `sqrt(d_head)` to keep the numbers from blowing up at higher dimensions.

### TODO 2. Compute scaled attention scores

Compute `scores = Q @ K.T / sqrt(d_head)` where `d_head` is the per-head dimension (here, 2).

In [ ]:
# TODO 2: scaled dot-product attention scores
d_head = Q.shape[1]  # 2

# Replace the None.
scores = None

print('scores shape:', scores.shape if scores is not None else 'TODO')
print('scores:'); print(scores)
# Expected shape: (5, 5). The cooked row should be highest on Jide, iyan, and peppers.

Once you have scores, visualise them as a heatmap. Each ROW is one token's outgoing attention pattern.

In [ ]:
def show_heatmap(M, title, ax=None, vmin=None, vmax=None, cmap='viridis'):
    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(M, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(tokens)), tokens, rotation=45, ha='right')
    ax.set_yticks(range(len(tokens)), tokens)
    ax.set_title(title)
    ax.set_ylabel('FROM (query)')
    ax.set_xlabel('TO (key)')
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            color = 'white' if (M[i,j] - (vmin or M.min())) / max(((vmax or M.max()) - (vmin or M.min())), 1e-9) < 0.5 else 'black'
            ax.text(j, i, f'{M[i,j]:.2f}', ha='center', va='center', color=color, fontsize=8)
    if own_fig:
        plt.colorbar(im, ax=ax)
        plt.tight_layout()
        plt.show()

if scores is not None:
    show_heatmap(scores, 'Raw attention scores (Q · K^T / sqrt(d_head))')

---
## Step 4. Softmax: from raw scores to a probability distribution

Raw scores can be any real numbers. We need each ROW to be a probability distribution: non-negative, summing to 1. Softmax does both.

The softmax of a row `s` is: `exp(s) / sum(exp(s))`. In practice, we subtract the max of the row first for numerical stability (so we never compute `exp(huge_number)`).

### TODO 3. Implement a numerically stable softmax that operates on rows

It should take a 2D array and return another 2D array where each ROW sums to 1.

In [ ]:
# TODO 3: row-wise stable softmax
def softmax(x):
    """Apply softmax to each row of x. Subtract the row max for stability."""
    # Your code here. Replace the body.
    return x  # placeholder

# Quick sanity test (uncomment after implementing):
# test = np.array([[1.0, 2.0, 3.0], [0.0, 0.0, 0.0]])
# print(softmax(test))
# # Expected first row: [0.090, 0.245, 0.665]
# # Expected second row: [0.333, 0.333, 0.333]

In [ ]:
# Apply softmax to your scores to get attention weights
if scores is not None:
    attn = softmax(scores)
    print('Each row of attn should sum to 1:')
    print(attn.sum(axis=1))
    print('\nattention weights:'); print(attn)
    show_heatmap(attn, 'Attention weights (after softmax)', vmin=0, vmax=1)

**Read the heatmap.**

- The **`cooked` row** is the most informative. Where is the attention going? It should be split between `Jide`, `iyan`, and `peppers`. The verb is looking at its arguments.
- The other rows are flatter, because their queries are weaker (they are not verbs).

If `cooked` distributes its attention roughly evenly across the three nouns, the head has learned something meaningful. In a real transformer, OTHER heads would specialise in OTHER patterns (subject-verb agreement, modifier-noun, etc.). We will simulate that in Step 7.

---
## Step 5. The output: a weighted sum of values

Once we have the attention weights, the output for each token is a weighted sum of all the value vectors. In matrix form, that is just `output = attn @ V`.

This is where information actually moves between tokens. After this step, the verb `cooked` will carry information about its arguments.

In [ ]:
if scores is not None:
    output = attn @ V
    print('output shape:', output.shape)
    print('\nValue V (before attention):'); print(V)
    print('\nOutput Y (after attention):'); print(output)

### TODO 4. Compare each token's V (before) to its Y (after)

Compute the per-token Euclidean distance between V and the output Y. Which token's representation moved the most?

Hint: `np.linalg.norm(output - V, axis=1)` gives you a 5-element array, one number per token.

In [ ]:
# TODO 4: per-token shift
if scores is not None:
    shifts = None  # replace with the per-token distance
    if shifts is not None:
        for tok, d in zip(tokens, shifts):
            print(f'{tok:8s} moved by {d:.3f}')

**Discuss.**  Which token moved most? Why? Was it the token that did the most attending, or a different token?

(There is a quirk hiding in our setup. Tokens with WEAK queries get averaged with everyone, so they can lose their distinctness even though they did not actively attend to anything. Tokens with STRONG queries pull in specific information. In our toy notebook, only `cooked` has a strong query, so this effect is exaggerated. In a real model, ALL tokens have learned strong queries, so the effect is less pronounced. This is a property of our specific hand-crafted projections, not of attention itself.)

---
## Step 6. Causal masking: no looking into the future

When a transformer is trained to predict the next token, every position must only see tokens that came BEFORE it (and itself). Otherwise the model could just copy the next token from the future. The fix is causal masking: before softmax, replace future positions with a very large negative number (here, `-1e9`). After `exp()`, those positions become essentially 0.

**One thing changes for this step only.** Real autoregressive transformers (like GPT) prepend a special **`<BOS>`** (beginning-of-sentence) token to every input. Without it, the very first content word at position 0 ends up attending only to itself under causal masking, which is technically correct but provides no useful context. By adding a `<BOS>` token in front, the first content word gets something to attend to other than itself, which is how real models actually behave.

So for this step only, the sentence becomes:

> `<BOS>, Jide, cooked, iyan, with, peppers` (6 tokens).

Everything after Step 6 returns to the 5-token sentence.

In [ ]:
# Step 6 only: add a <BOS> token in front of the sentence.
tokens_bos = ['<BOS>', 'Jide', 'cooked', 'iyan', 'with', 'peppers']

# <BOS> is given a small neutral embedding so it does not strongly resemble
# any of the four roles. It exists as a position-0 anchor.
X_bos = np.vstack([
    np.array([0.2, 0.2, 0.2, 0.2]),  # <BOS>: neutral across all dims
    X,                                # then the original 5 tokens
])

# Re-project with the SAME W matrices used earlier.
Q_bos = X_bos @ W_Q
K_bos = X_bos @ W_K
V_bos = X_bos @ W_V

print('X_bos shape:', X_bos.shape)
print('K_bos:'); print(K_bos)

### TODO 5. Build the causal mask and apply it to the 6-token sequence

1. Compute new scaled scores using `Q_bos @ K_bos.T / sqrt(d_head)`.
2. Build a (6, 6) array where `M[i, j] = 0` if `j <= i` (allowed) and `M[i, j] = -1e9` if `j > i` (future, masked out).
3. Add it to your scores.
4. Run softmax.
5. Visualise with the 6-token labels.

Hint: `np.triu(np.ones((6, 6)), k=1)` gives 1s strictly above the diagonal. Multiply by `-1e9`.

In [ ]:
# TODO 5: build the mask, apply it, and visualise.

# 1. Compute scaled scores for the 6-token sequence.
scores_bos = None

n = len(tokens_bos)

# 2. Build the causal mask: 0 on/below diagonal, -1e9 above.
causal_mask = None

if scores_bos is not None and causal_mask is not None:
    masked_scores = scores_bos + causal_mask
    masked_attn = softmax(masked_scores)
    print('masked attention rows sum to:', masked_attn.sum(axis=1))

    # Heatmap helper that uses the 6-token labels.
    def show_heatmap_bos(M, title, ax=None, vmin=None, vmax=None, cmap='viridis'):
        own_fig = ax is None
        if own_fig:
            fig, ax = plt.subplots(figsize=(6, 5))
        im = ax.imshow(M, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_xticks(range(len(tokens_bos)), tokens_bos, rotation=45, ha='right')
        ax.set_yticks(range(len(tokens_bos)), tokens_bos)
        ax.set_title(title)
        ax.set_ylabel('FROM (query)')
        ax.set_xlabel('TO (key)')
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                color = 'white' if (M[i,j] - (vmin or M.min())) / max(((vmax or M.max()) - (vmin or M.min())), 1e-9) < 0.5 else 'black'
                ax.text(j, i, f'{M[i,j]:.2f}', ha='center', va='center', color=color, fontsize=8)
        if own_fig:
            plt.colorbar(im, ax=ax)
            plt.tight_layout()
            plt.show()

    show_heatmap_bos(masked_attn, 'Causal-masked attention (with <BOS>)', vmin=0, vmax=1)

**Read the new heatmap.**

- The pattern is lower-triangular: future positions are dark.
- The `<BOS>` row attends 100% to itself, because it has nothing before it. That is the position-0 edge case, but it is harmless: `<BOS>` is an anchor token, not a content token, and we never read its output.
- The `Jide` row now splits between `<BOS>` and itself (roughly 50/50). Without `<BOS>`, Jide would have attended 100% to itself, which would have meant attention contributed nothing for the first content word. The `<BOS>` token gives Jide something to attend to and produces a more realistic representation.
- The `cooked` row leans strongly toward Jide (its subject), with smaller weight on `<BOS>` and itself.
- Later rows (`iyan`, `with`, `peppers`) spread across the visible past.

In an encoder there is no causal mask, so attention flows both ways. Decoders use causal masks at training time so the model learns to predict the next token honestly. The `<BOS>` token is a standard trick in autoregressive decoders to give position 0 a context anchor.

---
## Step 7. Multi-head attention: different heads, different patterns

So far we have ONE attention head. The pattern it learned (verb attends to nouns) is just one of many useful patterns.

Real transformers run many attention heads in parallel. Each head has its OWN `W_Q`, `W_K`, `W_V`, so each head asks a different question of the sentence. One head might learn subject-verb agreement. Another might learn co-reference. Another might learn topic coherence.

Below, a SECOND head has been hand-crafted to make subjects/objects find the verb (the inverse of head 1).

In [ ]:
# Head 2: subjects/objects find the verb
# Idea: K extracts action-ness, Q extracts person-ness or food-ness

W_Q2 = np.array([
    [0.5, 0.5],   # food-ness goes into Q
    [1.0, 1.0],   # person-ness goes into Q
    [0.0, 0.0],
    [0.0, 0.0],
])

W_K2 = np.array([
    [0.0, 0.0],
    [0.0, 0.0],
    [1.0, 1.0],   # action-ness is what K advertises
    [0.0, 0.0],
])

W_V2 = W_V  # reuse the same V projection for simplicity

Q2 = X @ W_Q2
K2 = X @ W_K2
V2 = X @ W_V2

print('Q2:'); print(Q2)
print('\nK2:'); print(K2)

### TODO 6. Compute the second head's attention pattern

Use the same recipe as before: scaled scores, then softmax. (No causal mask this time, just the regular pattern.)

In [ ]:
# TODO 6: head 2 attention
scores2 = None     # Q2 @ K2.T / sqrt(d_head)
attn2 = None       # softmax of scores2

if attn2 is not None:
    show_heatmap(attn2, 'Head 2: subjects/objects find the verb', vmin=0, vmax=1)

Now compare both heads side by side.

In [ ]:
if scores is not None and attn2 is not None:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    show_heatmap(attn, 'Head 1: verb finds its arguments', ax=axes[0], vmin=0, vmax=1)
    show_heatmap(attn2, 'Head 2: subjects/objects find the verb', ax=axes[1], vmin=0, vmax=1)
    plt.tight_layout()
    plt.show()

**Read both heatmaps.**

- Head 1: the brightest cells should be in the `cooked` row, columns `Jide`, `iyan`, `peppers`. The verb is looking at the nouns.
- Head 2: the brightest cells should be in the columns `cooked` (everyone is looking at the verb) and especially in the rows for `Jide`, `iyan`, `peppers`.

**What this is and is not.** Same input X. Same operation. Different `W_Q`, `W_K`. Different attention pattern. In our toy, Head 2 is essentially Head 1 with Q and K swapped, so it reverses the attention direction. In a real transformer, each head has its OWN independently learned W matrices, and the 8 to 16 heads collectively cover patterns ranging from local syntax to long-range coreference. Our two heads demonstrate that DIFFERENT W produces DIFFERENT attention; they do NOT demonstrate that heads automatically learn distinct linguistic relationships. That is a property of training, not of the attention mechanism itself.

In a real transformer, you would concatenate the outputs of both heads (and 6 or 14 more) and pass through a final projection `W_O` to combine them. We are stopping here.

---
## Final reflection

Spend the last five minutes here. Write your answers in the cell below.

### TODO 7. Explain attention without the words "Q", "K", "V"

Imagine a maths-strong but ML-new student asks: *what does attention actually do?* You have to answer in three or four sentences, in plain language, WITHOUT using the words "query", "key", or "value".

Write your answer in the cell below.

**Your answer (write below):**

_(double-click this cell to edit. Replace this line with your three to four sentences.)_

---
## What you just did

1. You built a 4D embedding for five tokens of an African-context sentence.
2. You projected those embeddings into Q, K, V.
3. You computed scaled attention scores.
4. You implemented a numerically stable softmax.
5. You produced an attention weight heatmap and read it.
6. You computed the attention output and measured how each token's representation shifted.
7. You applied causal masking and saw the heatmap go lower-triangular.
8. You ran a SECOND head with different projections and saw a completely different attention pattern emerge from the same input.

That is not a fragment of a transformer. That is the heart of it. Stack this op many times, throw an MLP between layers, add residual connections, and you have an LLM.

